<a href="https://colab.research.google.com/github/TheZaraKhan/ai-reliability/blob/main/benchmark/build_answer_key.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build the Toronto Bike Share benchmark answer key

This notebook rebuilds the reference answers for a fixed set of 40 questions
about Bike Share Toronto trips that started from January through March 2024.

I used AI assistance to draft questions and pandas code. The reference answers
are calculated from the official trip records using deterministic pandas
functions. The evaluation notebook uses the exported CSV to grade model outputs.

[City of Toronto Bike Share Ridership Data](https://open.toronto.ca/dataset/bike-share-toronto-ridership-data/)

## 1. Load the official Bike Share data

Load the City of Toronto’s 2024 ridership archive and keep trips that started
on or after January 1 and before April 1, 2024. Keep duplicate records and
unusual durations so the reference calculations can apply each question’s
cleaning rule.


In [4]:
import json
from pathlib import Path
from urllib.request import urlopen, urlretrieve
from zipfile import ZipFile

import pandas as pd

archive_path = Path("bikeshare-ridership-2024.zip")
ckan_url = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/"
    "package_show?id=bike-share-toronto-ridership-data"
)

# Find the official download URL through Toronto's catalogue.
# Reuse the ZIP if it is already in this Colab session.
if not archive_path.exists():
    with urlopen(ckan_url, timeout=60) as response:
        resources = json.load(response)["result"]["resources"]

    resource = next(item for item in resources if item["name"] == archive_path.name)
    urlretrieve(resource["url"], archive_path)

quarter_start = pd.Timestamp("2024-01-01")
quarter_end = pd.Timestamp("2024-04-01")

# Read the annual CSV in batches so only the selected quarter stays in memory.
quarter_chunks = []
with ZipFile(archive_path) as archive:
    with archive.open("bikeshare-ridership-2024.csv") as csv_file:
        for chunk in pd.read_csv(csv_file, chunksize=200_000):
            # Match the column names used by your reference functions.
            chunk.columns = (
                chunk.columns.str.lower()
                .str.replace(r"[^a-z0-9]+", "_", regex=True)
                .str.strip("_")
            )

            # Use the start timestamp to decide whether a trip belongs to Q1.
            chunk["start_time"] = pd.to_datetime(
                chunk["start_time"], format="%Y-%m-%d %H:%M:%S"
            )
            in_quarter = (
                chunk["start_time"].ge(quarter_start)
                & chunk["start_time"].lt(quarter_end)
            )
            quarter_chunks.append(chunk.loc[in_quarter].copy())

# Combine the selected rows and parse their end timestamps.
df = pd.concat(quarter_chunks, ignore_index=True)
df["end_time"] = pd.to_datetime(df["end_time"], format="%Y-%m-%d %H:%M:%S")

display(df.head())

,trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model
0,26682740,528,7041,2024-01-01 00:00:08,Edward St / Yonge St,7130.0,2024-01-01 00:08:56,Davenport Rd / Bedford Rd,7855,Casual,EFIT G5
1,26682742,1924,7245,2024-01-01 00:02:04,Tecumseth St / Queen St W - SMART,7259.0,2024-01-01 00:34:08,Lower Spadina Ave / Lake Shore Blvd W,2453,Casual,ICONIC
2,26682743,2741,7052,2024-01-01 00:03:25,Wellington St W / Bay St,7366.0,2024-01-01 00:49:06,Fort York Blvd / Bathurst St - SMART,7332,Member,ICONIC
3,26682744,353,7041,2024-01-01 00:03:39,Edward St / Yonge St,7418.0,2024-01-01 00:09:32,College Park - Yonge St Entrance,782,Member,ICONIC
4,26682746,2118,7721,2024-01-01 00:04:44,Widmer St / King St W,7721.0,2024-01-01 00:40:02,Widmer St / King St W,1008,Member,ICONIC


## 2 — Questions and answer key



### 2.1 · Define the questions

The evaluation uses 20 questions about official Toronto Bike Share trips from
January through March 2024. It covers simple lookups, multi-step calculations,
sensitivity to data cleaning, ambiguous wording, and questions the dataset
cannot answer.

The model-facing question table contains question IDs, categories, and prompts.
The reference answers are computed and reviewed separately.

In [5]:
# Stage 1 must have loaded the official data in this Colab session.
if (
    "df" not in globals()
    or globals().get("USE_FIXTURE") is not False
    or "bike_model" not in df.columns
):
    raise RuntimeError("Run Stage 1 with USE_FIXTURE = False first.")

# Each row is one question. The ID connects the question to its oracle.
questions = pd.DataFrame(
    [
        ("Q01", "simple", "How many trip records are in the quarter?"),
        ("Q02", "simple", "What is the shortest recorded trip duration in seconds?"),
        ("Q03", "simple", "What is the longest recorded trip duration in seconds?"),
        ("Q04", "simple", "How many trips have a recorded duration under 60 seconds?"),
        ("Q05", "simple", "How many trips have a missing end station ID?"),
        ("Q06", "simple", "How many distinct non-missing bike IDs appear in the data?"),
        ("Q07", "simple", "How many trips have user type Member?"),
        ("Q08", "simple", "What is the median recorded trip duration in seconds?"),

        ("Q09", "multi_step", "Which calendar month had the most trip starts? Give the month as YYYY-MM. If tied, give the earliest month."),
        ("Q10", "multi_step", "What percentage of March trip records have user type Casual? Use all March trip records as the denominator and report on a 0–100 scale."),
        ("Q11", "multi_step", "Among trips with both station IDs present, how many started and ended at the same station?"),
        ("Q12", "multi_step", "What is the median recorded trip duration in seconds for Member trips that started on Saturday or Sunday?"),
        ("Q13", "multi_step", "Which start station ID had the most Casual trip starts in March? Exclude missing start station IDs. If tied, give the lowest station ID."),

        ("Q14", "messy_data", "Give the mean recorded duration in seconds using all rows, and again using only trips with durations from 60 through 86,400 seconds inclusive. Label the two values raw and cleaned."),
        ("Q15", "messy_data", "Give the number of trip records before and after removing exact duplicate rows. Label the two counts raw and cleaned."),
        ("Q16", "messy_data", "Give the percentage of trip records with a missing end time before and after excluding records with a missing end station ID. Report both on a 0–100 scale and label them raw and cleaned."),

        ("Q17", "ambiguous", "How many trips happened in March? State which timestamp you used."),
        ("Q18", "ambiguous", "What was the typical trip duration in seconds? State which statistic you used."),

        ("Q19", "unanswerable", "What was the average age of riders in the quarter?"),
        ("Q20", "unanswerable", "What was the total distance travelled by riders in the quarter, in kilometres?"),
    ],
    columns=["question_id", "category", "question"],
)

display(questions)

,question_id,category,question
0,Q01,simple,How many trip records are in the quarter?
1,Q02,simple,What is the shortest recorded trip duration in...
2,Q03,simple,What is the longest recorded trip duration in ...
3,Q04,simple,How many trips have a recorded duration under ...
4,Q05,simple,How many trips have a missing end station ID?
5,Q06,simple,How many distinct non-missing bike IDs appear ...
6,Q07,simple,How many trips have user type Member?
7,Q08,simple,What is the median recorded trip duration in s...
8,Q09,multi_step,Which calendar month had the most trip starts?...
9,Q10,multi_step,What percentage of March trip records have use...


### 2.2 · Define the pandas oracles

Each question has a reference function that calculates its answer from the
official trip records. The month and weekend calculations use the recorded
timestamps as loaded in Stage 1.

For the cleaning questions, the answer key shows both raw and cleaned values.
The cleaning rules apply only to those questions. They leave the underlying
trip table intact.

For the ambiguous questions, we record the interpretations accepted in this
benchmark. Each answer must identify its interpretation.

In [6]:
# Each function receives the same DataFrame.
# The simple questions each use one main pandas operation.

def q01(data: pd.DataFrame) -> int:
    return len(data)


def q02(data: pd.DataFrame) -> int:
    return int(data["trip_duration"].min())


def q03(data: pd.DataFrame) -> int:
    return int(data["trip_duration"].max())


def q04(data: pd.DataFrame) -> int:
    return int(data["trip_duration"].lt(60).sum())


def q05(data: pd.DataFrame) -> int:
    return int(data["end_station_id"].isna().sum())


def q06(data: pd.DataFrame) -> int:
    return int(data["bike_id"].nunique(dropna=True))


def q07(data: pd.DataFrame) -> int:
    return int(data["user_type"].eq("Member").sum())


def q08(data: pd.DataFrame) -> float:
    return float(data["trip_duration"].median())


# The multi-step questions combine filtering, grouping, or comparison.

def q09(data: pd.DataFrame) -> str:
    # Sort the months first so idxmax picks the earliest month if counts tie.
    months = data["start_time"].dt.to_period("M")
    counts = months.value_counts().sort_index()
    return str(counts.idxmax())


def q10(data: pd.DataFrame) -> float:
    march = data.loc[data["start_time"].dt.month.eq(3)]
    # Count Casual trips and divide by all March trip records.
    return 100 * int(march["user_type"].eq("Casual").sum()) / len(march)


def q11(data: pd.DataFrame) -> int:
    # Compare station IDs only for records with both endpoints present.
    trips = data.dropna(subset=["start_station_id", "end_station_id"])
    return int(trips["start_station_id"].eq(trips["end_station_id"]).sum())


def q12(data: pd.DataFrame) -> float:
    # pandas numbers the days Monday = 0 through Sunday = 6.
    weekend = data["start_time"].dt.dayofweek.isin([5, 6])
    member = data["user_type"].eq("Member")
    return float(data.loc[weekend & member, "trip_duration"].median())


def q13(data: pd.DataFrame) -> int:
    march = data["start_time"].dt.month.eq(3)
    casual = data["user_type"].eq("Casual")
    stations = data.loc[march & casual, "start_station_id"].dropna()
    # Sorting the station IDs first gives the lowest ID if counts tie.
    return int(stations.value_counts().sort_index().idxmax())


# For messy-data questions, return both answers under explicit labels.

def q14(data: pd.DataFrame) -> dict[str, float]:
    duration = data["trip_duration"]
    cleaned = duration[duration.between(60, 86_400, inclusive="both")]
    return {"raw": float(duration.mean()), "cleaned": float(cleaned.mean())}


def q15(data: pd.DataFrame) -> dict[str, int]:
    # Exact duplicates must match across every column.
    return {"raw": len(data), "cleaned": len(data.drop_duplicates())}


def q16(data: pd.DataFrame) -> dict[str, float]:
    with_end_station = data.loc[data["end_station_id"].notna()]
    return {
        "raw": 100 * float(data["end_time"].isna().mean()),
        "cleaned": 100 * float(with_end_station["end_time"].isna().mean()),
    }


# Each key names one acceptable interpretation for this benchmark.

def q17(data: pd.DataFrame) -> dict[str, int]:
    # Both counts use the same Q1 extract, which Stage 1 selected by start time.
    return {
        "start_time": int(data["start_time"].dt.month.eq(3).sum()),
        "end_time": int(data["end_time"].dt.month.eq(3).sum()),
    }


def q18(data: pd.DataFrame) -> dict[str, float]:
    duration = data["trip_duration"]
    return {"mean": float(duration.mean()), "median": float(duration.median())}


# None means the required information is unavailable in this dataset.

def q19(data: pd.DataFrame) -> None:
    return None  # Trip records do not contain rider ages.


def q20(data: pd.DataFrame) -> None:
    return None  # Trip records do not contain distance travelled.

### 2.3 · Review and verify the answer key

The review table shows each question, the pandas function that answers it,
and the reference answer computed from the official data.

The checks confirm the category counts, verify that every question has an
oracle, and cross-check selected calculations against independent totals.
For questions about unavailable information, the review table explains why
the dataset cannot answer.

In [9]:
import inspect
import math

# Explicitly connect each question ID to its function.
oracles = {
    "Q01": q01, "Q02": q02, "Q03": q03, "Q04": q04,
    "Q05": q05, "Q06": q06, "Q07": q07, "Q08": q08,
    "Q09": q09, "Q10": q10, "Q11": q11, "Q12": q12, "Q13": q13,
    "Q14": q14, "Q15": q15, "Q16": q16,
    "Q17": q17, "Q18": q18, "Q19": q19, "Q20": q20,
}

notes = {
    "Q14": "Keep durations from 60 through 86,400 seconds, inclusive.",
    "Q15": "Drop exact duplicates across all columns.",
    "Q16": "Keep only records with a known end station ID.",
    "Q17": "Accept March starts or March ends within the Q1 extract, with the timestamp stated.",
    "Q18": "Accept the mean or median of recorded durations, with the statistic stated.",
    "Q19": "The data does not contain rider ages.",
    "Q20": "The data does not contain distance travelled.",
}

# Check that all 20 questions are present and mapped exactly once.
assert questions["question_id"].is_unique
assert set(questions["question_id"]) == set(oracles)
assert questions["category"].value_counts().to_dict() == {
    "simple": 8,
    "multi_step": 5,
    "messy_data": 3,
    "ambiguous": 2,
    "unanswerable": 2,
}

# Compute every truth by calling the corresponding function on the real DataFrame.
truth_by_id = {
    question_id: oracle(df)
    for question_id, oracle in oracles.items()
}

# Cross-check a few answers. The first two use Stage 1's quality table.
stage1_checks = quality.set_index("check")["value"]
assert truth_by_id["Q01"] == stage1_checks.loc["Rows"]
assert truth_by_id["Q04"] == stage1_checks.loc["Trips under 60 seconds"]
assert truth_by_id["Q15"]["raw"] - truth_by_id["Q15"]["cleaned"] == (
    stage1_checks.loc["Duplicate rows after the first"]
)

# Every answerable question should return finite numbers or a month string.
# For the dictionary answers, check each value inside the dictionary.
for question_id, answer in truth_by_id.items():
    if question_id in {"Q19", "Q20"}:
        assert answer is None, question_id
        continue

    values = answer.values() if isinstance(answer, dict) else [answer]
    assert all(
        isinstance(value, str)
        or (isinstance(value, (int, float)) and math.isfinite(value))
        for value in values
    ), question_id

# Build a separate table for your review, including the actual oracle code.
review = questions.copy()
review["oracle"] = review["question_id"].map(
    lambda question_id: inspect.getsource(oracles[question_id]).strip()
)
review["computed_truth"] = review["question_id"].map(truth_by_id)
review["rule_or_note"] = review["question_id"].map(notes).fillna("")

display(
    review.style
    .set_properties(subset=["oracle"], **{"white-space": "pre-wrap", "text-align": "left"})
    .set_properties(subset=["question", "rule_or_note"], **{"text-align": "left"})
)

print("Stage 2 checks passed for all 20 questions.")

,question_id,category,question,oracle,computed_truth,rule_or_note
0,Q01,simple,How many trip records are in the quarter?,def q01(data: pd.DataFrame) -> int: return len(data),779266,
1,Q02,simple,What is the shortest recorded trip duration in seconds?,"def q02(data: pd.DataFrame) -> int: return int(data[""trip_duration""].min())",0,
2,Q03,simple,What is the longest recorded trip duration in seconds?,"def q03(data: pd.DataFrame) -> int: return int(data[""trip_duration""].max())",2320126,
3,Q04,simple,How many trips have a recorded duration under 60 seconds?,"def q04(data: pd.DataFrame) -> int: return int(data[""trip_duration""].lt(60).sum())",1572,
4,Q05,simple,How many trips have a missing end station ID?,"def q05(data: pd.DataFrame) -> int: return int(data[""end_station_id""].isna().sum())",283,
5,Q06,simple,How many distinct non-missing bike IDs appear in the data?,"def q06(data: pd.DataFrame) -> int: return int(data[""bike_id""].nunique(dropna=True))",6704,
6,Q07,simple,How many trips have user type Member?,"def q07(data: pd.DataFrame) -> int: return int(data[""user_type""].eq(""Member"").sum())",708158,
7,Q08,simple,What is the median recorded trip duration in seconds?,"def q08(data: pd.DataFrame) -> float: return float(data[""trip_duration""].median())",597.000000,
8,Q09,multi_step,"Which calendar month had the most trip starts? Give the month as YYYY-MM. If tied, give the earliest month.","def q09(data: pd.DataFrame) -> str: # Sort the months first so idxmax picks the earliest month if counts tie. months = data[""start_time""].dt.to_period(""M"") counts = months.value_counts().sort_index() return str(counts.idxmax())",2024-03,
9,Q10,multi_step,What percentage of March trip records have user type Casual? Use all March trip records as the denominator and report on a 0–100 scale.,"def q10(data: pd.DataFrame) -> float: march = data.loc[data[""start_time""].dt.month.eq(3)] # Count Casual trips and divide by all March trip records. return 100 * int(march[""user_type""].eq(""Casual"").sum()) / len(march)",10.565779,


Stage 2 checks passed for all 20 questions.


In [11]:
# Keep these questions in a separate table while you review their wording.
additional_questions = pd.DataFrame(
    [
        # Simple questions
        ("Q21", "simple",
         "How many trip records have the bicycle model EFIT G5?"),
        ("Q22", "simple",
         "How many distinct non-missing bicycle model labels appear?"),
        ("Q23", "simple",
         "How many trips started on February 29, 2024? Use start_time."),
        ("Q24", "simple",
         "How many trips started from 07:00 inclusive to 08:00 exclusive?"),
        ("Q25", "simple",
         "How many trips have a recorded duration of exactly 60 seconds?"),
        ("Q26", "simple",
         "How many distinct non-missing end station IDs appear?"),
        ("Q27", "simple",
         "How many trips started on a Sunday?"),
        ("Q28", "simple",
         "What is the 90th percentile of recorded trip duration in seconds? "
         "Use linear interpolation."),

        # Multi-step questions
        ("Q29", "multi_step",
         "What was the percentage change in trip starts from January to March "
         "2024? Use January's count as the baseline and report the signed percentage."),
        ("Q30", "multi_step",
         "Which start station ID had the most trip starts from 07:00 inclusive "
         "to 10:00 exclusive on Mondays through Fridays? Exclude missing IDs. "
         "If tied, give the lowest station ID."),
        ("Q31", "multi_step",
         "Among records with a non-missing bike ID and user type Member or Casual, "
         "how many distinct bike IDs appear in both user types?"),
        ("Q32", "multi_step",
         "Which calendar date in February 2024 had the most Casual trip starts? "
         "Give the date as YYYY-MM-DD. If tied, give the earliest date."),
        ("Q33", "multi_step",
         "For trips that started at station 7016 and ended at a different station, "
         "which end station ID appears most often? Exclude missing end station IDs. "
         "If tied, give the lowest end station ID."),

        # Messy-data questions
        ("Q34", "messy_data",
         "Compute the percentage of trip records that returned to their start station. "
         "For raw, divide matching non-missing station pairs by all trip records. "
         "For cleaned, divide the same count by records with both station IDs present. "
         "Report both on a 0–100 scale and label them raw and cleaned."),
        ("Q35", "messy_data",
         "How many distinct non-missing start station names appear before and after "
         "stripping leading and trailing whitespace and converting names to lowercase? "
         "If normalization produces an empty name, exclude it from the cleaned count. "
         "Label the counts raw and cleaned."),
        ("Q36", "messy_data",
         "Among trips with both timestamps present, how many have a recorded duration "
         "that differs from end_time minus start_time? For raw, count any nonzero "
         "absolute difference. For cleaned, count only absolute differences greater "
         "than one second. Label the counts raw and cleaned."),

        # Ambiguous questions
        ("Q37", "ambiguous",
         "What was the busiest station in Q1? Report the station ID and state whether "
         "you counted departures, arrivals, or both. Exclude missing station IDs "
         "and give the lowest ID if tied."),
        ("Q38", "ambiguous",
         "What was the busiest day in Q1? State whether you mean a single calendar "
         "date or a day of the week. Use trip starts. If tied, pick the earliest "
         "date or the earliest weekday in Monday-to-Sunday order."),

        # Unanswerable questions
        ("Q39", "unanswerable",
         "How many attempted rentals failed because no bike was available "
         "at the requested station?"),
        ("Q40", "unanswerable",
         "How many distinct people took a Bike Share trip during the quarter?"),
    ],
    columns=["question_id", "category", "question"],
)

display(additional_questions)

,question_id,category,question
0,Q21,simple,How many trip records have the bicycle model E...
1,Q22,simple,How many distinct non-missing bicycle model la...
2,Q23,simple,"How many trips started on February 29, 2024? U..."
3,Q24,simple,How many trips started from 07:00 inclusive to...
4,Q25,simple,How many trips have a recorded duration of exa...
5,Q26,simple,How many distinct non-missing end station IDs ...
6,Q27,simple,How many trips started on a Sunday?
7,Q28,simple,What is the 90th percentile of recorded trip d...
8,Q29,multi_step,What was the percentage change in trip starts ...
9,Q30,multi_step,Which start station ID had the most trip start...


In [12]:
def q21(data: pd.DataFrame) -> int:
    return int(data["bike_model"].eq("EFIT G5").sum())

def q22(data: pd.DataFrame) -> int:
    return int(data["bike_model"].dropna().nunique())

def q23(data: pd.DataFrame) -> int:
    return int(data["start_time"].dt.normalize().eq(pd.Timestamp("2024-02-29")).sum())

def q24(data: pd.DataFrame) -> int:
    hours = data["start_time"].dt.hour
    return int(hours.eq(7).sum())

def q25(data: pd.DataFrame) -> int:
    return int(data["trip_duration"].eq(60).sum())

def q26(data: pd.DataFrame) -> int:
    return int(data["end_station_id"].dropna().nunique())

def q27(data: pd.DataFrame) -> int:
    return int(data["start_time"].dt.dayofweek.eq(6).sum())

def q28(data: pd.DataFrame) -> float:
    return float(data["trip_duration"].quantile(0.9, interpolation="linear"))

def q29(data: pd.DataFrame) -> float:
    starts = data["start_time"].dt.month
    jan = starts.eq(1).sum()
    mar = starts.eq(3).sum()
    return float(((mar - jan) / jan) * 100)

def q30(data: pd.DataFrame) -> int:
    starts = data["start_time"]
    is_weekday = starts.dt.dayofweek.lt(5)
    is_hour_range = starts.dt.hour.between(7, 9, inclusive="both")
    stations = data.loc[is_weekday & is_hour_range, "start_station_id"].dropna()
    return int(stations.value_counts().sort_index().idxmax())

def q31(data: pd.DataFrame) -> int:
    valid = data.dropna(subset=["bike_id"]).loc[data["user_type"].isin(["Member", "Casual"])]
    groups = valid.groupby("bike_id")["user_type"].nunique()
    return int((groups == 2).sum())

def q32(data: pd.DataFrame) -> str:
    feb = data["start_time"].dt.month.eq(2)
    casual = data["user_type"].eq("Casual")
    dates = data.loc[feb & casual, "start_time"].dt.date
    return str(dates.value_counts().sort_index().idxmax())

def q33(data: pd.DataFrame) -> int:
    st_7016 = data["start_station_id"].eq(7016)
    diff_station = data["end_station_id"].ne(7016)
    endpoints = data.loc[st_7016 & diff_station, "end_station_id"].dropna()
    return int(endpoints.value_counts().sort_index().idxmax())

def q34(data: pd.DataFrame) -> dict[str, float]:
    matching = data["start_station_id"].eq(data["end_station_id"]) & data["start_station_id"].notna()
    num_matching = matching.sum()
    raw_pct = (num_matching / len(data)) * 100
    valid_records = data["start_station_id"].notna() & data["end_station_id"].notna()
    cleaned_pct = (num_matching / valid_records.sum()) * 100
    return {"raw": float(raw_pct), "cleaned": float(cleaned_pct)}

def q35(data: pd.DataFrame) -> dict[str, int]:
    raw_names = data["start_station_name"].dropna()
    raw_count = raw_names.nunique()
    cleaned_names = raw_names.str.strip().str.lower()
    cleaned_names = cleaned_names[cleaned_names.ne("")]
    return {"raw": int(raw_count), "cleaned": int(cleaned_names.nunique())}

def q36(data: pd.DataFrame) -> dict[str, int]:
    valid = data.dropna(subset=["start_time", "end_time"])
    calc_duration = (valid["end_time"] - valid["start_time"]).dt.total_seconds()
    diff = (valid["trip_duration"] - calc_duration).abs()
    return {"raw": int(diff.ne(0).sum()), "cleaned": int(diff.gt(1).sum())}

def q37(data: pd.DataFrame) -> dict[str, int]:
    departures = data["start_station_id"].dropna()
    arrivals = data["end_station_id"].dropna()
    both = pd.concat([departures, arrivals])
    return {
        "departures": int(departures.value_counts().sort_index().idxmax()),
        "arrivals": int(arrivals.value_counts().sort_index().idxmax()),
        "both": int(both.value_counts().sort_index().idxmax())
    }

def q38(data: pd.DataFrame) -> dict[str, str]:
    starts = data["start_time"]
    date_counts = starts.dt.date.value_counts().sort_index()
    day_counts = starts.dt.day_name().value_counts()
    days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    day_counts = day_counts.reindex(days_order)
    return {
        "calendar_date": str(date_counts.idxmax()),
        "day_of_week": str(day_counts.idxmax())
    }

def q39(data: pd.DataFrame) -> None:
    return None

def q40(data: pd.DataFrame) -> None:
    return None

In [13]:
import inspect

additional_oracles = {
    "Q21": q21, "Q22": q22, "Q23": q23, "Q24": q24, "Q25": q25,
    "Q26": q26, "Q27": q27, "Q28": q28, "Q29": q29, "Q30": q30,
    "Q31": q31, "Q32": q32, "Q33": q33, "Q34": q34, "Q35": q35,
    "Q36": q36, "Q37": q37, "Q38": q38, "Q39": q39, "Q40": q40
}

# Map oracle code as text into a new column
additional_questions["oracle_code"] = additional_questions["question_id"].map(
    lambda q_id: inspect.getsource(additional_oracles[q_id]).strip()
)

# Compute answers for verification
additional_questions["computed_truth"] = additional_questions["question_id"].map(
    lambda q_id: additional_oracles[q_id](df)
)

display(additional_questions)

,question_id,category,question,oracle_code,computed_truth
0,Q21,simple,How many trip records have the bicycle model E...,def q21(data: pd.DataFrame) -> int:\n retur...,22713
1,Q22,simple,How many distinct non-missing bicycle model la...,def q22(data: pd.DataFrame) -> int:\n retur...,3
2,Q23,simple,"How many trips started on February 29, 2024? U...",def q23(data: pd.DataFrame) -> int:\n retur...,8823
3,Q24,simple,How many trips started from 07:00 inclusive to...,def q24(data: pd.DataFrame) -> int:\n hours...,29156
4,Q25,simple,How many trips have a recorded duration of exa...,def q25(data: pd.DataFrame) -> int:\n retur...,30
5,Q26,simple,How many distinct non-missing end station IDs ...,def q26(data: pd.DataFrame) -> int:\n retur...,798
6,Q27,simple,How many trips started on a Sunday?,def q27(data: pd.DataFrame) -> int:\n retur...,83238
7,Q28,simple,What is the 90th percentile of recorded trip d...,def q28(data: pd.DataFrame) -> float:\n ret...,1485.0
8,Q29,multi_step,What was the percentage change in trip starts ...,def q29(data: pd.DataFrame) -> float:\n sta...,52.584208
9,Q30,multi_step,Which start station ID had the most trip start...,def q30(data: pd.DataFrame) -> int:\n start...,7033


In [15]:
import inspect

# Prepare the first set of 20 questions in the same format
questions_copy = questions.copy()
questions_copy["oracle_code"] = questions_copy["question_id"].map(
    lambda q_id: inspect.getsource(oracles[q_id]).strip()
)
questions_copy["computed_truth"] = questions_copy["question_id"].map(truth_by_id)

# Combine both DataFrames
all_questions = pd.concat([questions_copy, additional_questions], ignore_index=True)

# Save the combined 40 questions to a CSV file
csv_path = "/content/questions.csv"
all_questions.to_csv(csv_path, index=False)
print(f"Saved all 40 questions to: {csv_path}")

# Display the top and bottom of the unified DataFrame to verify
display(all_questions.head(5))
display(all_questions.tail(5))

Saved all 40 questions to: /content/questions.csv


,question_id,category,question,oracle_code,computed_truth
0,Q01,simple,How many trip records are in the quarter?,def q01(data: pd.DataFrame) -> int:\n retur...,779266
1,Q02,simple,What is the shortest recorded trip duration in...,def q02(data: pd.DataFrame) -> int:\n retur...,0
2,Q03,simple,What is the longest recorded trip duration in ...,def q03(data: pd.DataFrame) -> int:\n retur...,2320126
3,Q04,simple,How many trips have a recorded duration under ...,def q04(data: pd.DataFrame) -> int:\n retur...,1572
4,Q05,simple,How many trips have a missing end station ID?,def q05(data: pd.DataFrame) -> int:\n retur...,283


,question_id,category,question,oracle_code,computed_truth
35,Q36,messy_data,"Among trips with both timestamps present, how ...","def q36(data: pd.DataFrame) -> dict[str, int]:...","{'raw': 64, 'cleaned': 64}"
36,Q37,ambiguous,What was the busiest station in Q1? Report the...,"def q37(data: pd.DataFrame) -> dict[str, int]:...","{'departures': 7006, 'arrivals': 7033, 'both':..."
37,Q38,ambiguous,What was the busiest day in Q1? State whether ...,"def q38(data: pd.DataFrame) -> dict[str, str]:...","{'calendar_date': '2024-03-13', 'day_of_week':..."
38,Q39,unanswerable,How many attempted rentals failed because no b...,def q39(data: pd.DataFrame) -> None:\n retu...,None
39,Q40,unanswerable,How many distinct people took a Bike Share tri...,def q40(data: pd.DataFrame) -> None:\n retu...,None
